<a href="https://colab.research.google.com/github/cat1625/MLPF/blob/model/MLPF_UrbanFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#setup
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import warnings
warnings.filterwarnings('ignore')

# Project structure
BASE_DIR = "/content/drive/MyDrive/Shared drives/UrbanFlow"
RAW_DATA_DIR = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data/processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

for folder in [PROCESSED_DATA_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("✅ Environment ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready


In [ ]:
!pip install -q pyarrow fastparquet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans, DBSCAN

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded")

✅ Libraries loaded


In [ ]:
# Extract all .parquet.zip files
"""zip_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".parquet.zip")]
print(f"Found {len(zip_files)} zip files")

for zip_name in zip_files:
    zip_path = os.path.join(RAW_DATA_DIR, zip_name)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(RAW_DATA_DIR)

print("✅ Extraction complete")"""

'zip_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".parquet.zip")]\nprint(f"Found {len(zip_files)} zip files")\n\nfor zip_name in zip_files:\n    zip_path = os.path.join(RAW_DATA_DIR, zip_name)\n    with zipfile.ZipFile(zip_path, \'r\') as zip_ref:\n        zip_ref.extractall(RAW_DATA_DIR)\n\nprint("✅ Extraction complete")'

In [ ]:
#Loading with sampling

# Find parquet files
parquet_files = sorted([
    os.path.join(RAW_DATA_DIR, f)
    for f in os.listdir(RAW_DATA_DIR)
    if f.endswith(".parquet") and not f.endswith(".zip")
])

print(f"Found {len(parquet_files)} parquet files")
print("\n🎯 Loading Strategy: 30% stratified sampling per month")
print("Purpose: Preserve temporal patterns while ensuring memory safety\n")

# Essential columns (based on your data structure)
columns_to_keep = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count"
]

df_list = []
monthly_stats = []

for file in parquet_files:
    month_name = os.path.basename(file)
    print(f"📁 {month_name}")

    # Load month
    df = pd.read_parquet(file, columns=columns_to_keep)
    original_rows = len(df)

    # Stratified sampling by hour
    df['temp_hour'] = pd.to_datetime(df['tpep_pickup_datetime']).dt.hour
    df = df.groupby('temp_hour', group_keys=False).apply(
        lambda x: x.sample(frac=0.3, random_state=42)
    )
    df = df.drop(columns=['temp_hour'])

    sampled_rows = len(df)

    print(f"   Original: {original_rows:,} → Sampled: {sampled_rows:,}")
    print(f"   Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n")

    monthly_stats.append({
        'month': month_name,
        'original': original_rows,
        'sampled': sampled_rows
    })

    df_list.append(df)
    del df
    gc.collect()

# Summary
stats_df = pd.DataFrame(monthly_stats)
print("="*60)
print(f"Total original: {stats_df['original'].sum():,}")
print(f"Total sampled: {stats_df['sampled'].sum():,}")
print(f"Sampling rate: {stats_df['sampled'].sum() / stats_df['original'].sum() * 100:.1f}%")
print("="*60)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Shared drives/UrbanFlow/data/raw'

In [ ]:
#Concatenation & Initial Processing
print("\n🔄 Concatenating data...")

taxi_df = pd.concat(df_list, ignore_index=True)
del df_list
gc.collect()

print(f"✅ Combined dataset: {len(taxi_df):,} rows")
print(f"Memory: {taxi_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Convert datetime
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'])
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'])

taxi_df.head()

In [ ]:
# Load zone lookup
zone_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "taxi_zone_lookup (1).csv"))

print("Merging Pickup Zones...")
# Merge for Pickup
taxi_df = taxi_df.merge(
    zone_df[['LocationID', 'Borough', 'Zone']],
    how="left",
    left_on="PULocationID",
    right_on="LocationID"
)

# Rename the newly added columns
taxi_df.rename(columns={
    "Borough": "pickup_borough",
    "Zone": "pickup_zone"
}, inplace=True)

# Remove the redundant LocationID column if it exists
if 'LocationID' in taxi_df.columns:
    taxi_df.drop(columns=['LocationID'], inplace=True)

print("Merging Dropoff Zones...")
# Merge for Dropoff
taxi_df = taxi_df.merge(
    zone_df[['LocationID', 'Borough', 'Zone']],
    how="left",
    left_on="DOLocationID",
    right_on="LocationID"
)

# Rename the newly added columns
taxi_df.rename(columns={
    "Borough": "dropoff_borough",
    "Zone": "dropoff_zone"
}, inplace=True)

# Clean up redundant LocationID again
if 'LocationID' in taxi_df.columns:
    taxi_df.drop(columns=['LocationID'], inplace=True)

print("✅ Zone data merged successfully")
print(f"Current columns: {taxi_df.columns.tolist()}")

In [ ]:
#Feature Engineering

print("🔧 Engineering features...")

# Trip duration (TARGET VARIABLE)
taxi_df['trip_duration_minutes'] = (
    taxi_df['tpep_dropoff_datetime'] - taxi_df['tpep_pickup_datetime']
).dt.total_seconds() / 60

# Temporal features
taxi_df['pickup_hour'] = taxi_df['tpep_pickup_datetime'].dt.hour
taxi_df['pickup_day'] = taxi_df['tpep_pickup_datetime'].dt.day
taxi_df['pickup_dayofweek'] = taxi_df['tpep_pickup_datetime'].dt.dayofweek
taxi_df['pickup_month'] = taxi_df['tpep_pickup_datetime'].dt.month
taxi_df['pickup_date'] = taxi_df['tpep_pickup_datetime'].dt.date

# Binary features
taxi_df['is_weekend'] = taxi_df['pickup_dayofweek'].isin([5, 6]).astype(int)
taxi_df['is_rush_hour'] = taxi_df['pickup_hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

# Speed (derived feature)
taxi_df['avg_speed_mph'] = np.where(
    taxi_df['trip_duration_minutes'] > 0,
    (taxi_df['trip_distance'] / (taxi_df['trip_duration_minutes'] / 60)),
    0
)

print("✅ Features created")
taxi_df.head()

In [ ]:
print("Cleaning data...")

initial_rows = len(taxi_df)

# Check if columns exist before dropping
required_cols = ['pickup_borough', 'dropoff_borough']
existing_cols = [c for c in required_cols if c in taxi_df.columns]

if len(existing_cols) == len(required_cols):
    # Remove invalid trips
    taxi_df = taxi_df[
        (taxi_df['trip_distance'] > 0) & (taxi_df['trip_distance'] < 100) &
        (taxi_df['fare_amount'] > 0) & (taxi_df['fare_amount'] < 500) &
        (taxi_df['trip_duration_minutes'] > 1) & (taxi_df['trip_duration_minutes'] < 180) &
        (taxi_df['passenger_count'] >= 1) & (taxi_df['passenger_count'] <= 6)
    ].dropna(subset=['pickup_borough', 'dropoff_borough'])

    print(f"Removed {initial_rows - len(taxi_df):,} rows ({(initial_rows - len(taxi_df))/initial_rows*100:.1f}%)")
    print(f"Final clean dataset: {len(taxi_df):,} rows")
else:
    print(f"❌ Error: Missing columns {set(required_cols) - set(existing_cols)}")

In [ ]:
import zipfile

print("🌦️ Processing Weather data...")

weather_csv_path = os.path.join(RAW_DATA_DIR, "NYC_Weather_2016_2022.csv")
weather_zip_path = os.path.join(RAW_DATA_DIR, "NYC_Weather_2016_2022.csv.zip")

# 1. Extraction Check
if not os.path.exists(weather_csv_path):
    if os.path.exists(weather_zip_path):
        with zipfile.ZipFile(weather_zip_path, 'r') as zip_ref:
            zip_ref.extractall(RAW_DATA_DIR)
        print("✅ Extraction complete.")
    else:
        raise FileNotFoundError("Weather file not found.")

# 2. Load weather
weather_df = pd.read_csv(weather_csv_path)

# 3. DYNAMIC COLUMN DETECTION
try:
    temp_col = [col for col in weather_df.columns if 'temperature' in col.lower() or 'temp' in col.lower()][0]
    precip_col = [col for col in weather_df.columns if 'precipitation' in col.lower() or 'precip' in col.lower()][0]
    time_col = [col for col in weather_df.columns if 'time' in col.lower() or 'date' in col.lower()][0]
    print(f"🔍 Found columns: Time='{time_col}', Temp='{temp_col}', Precip='{precip_col}'")
except IndexError:
    print("❌ Could not detect columns. Available:", weather_df.columns.tolist())
    raise KeyError("Weather columns not found.")

# 4. Standardize and Filter
weather_df[time_col] = pd.to_datetime(weather_df[time_col])
weather_2021 = weather_df[weather_df[time_col].dt.year == 2021].copy()
weather_2021['date_only'] = weather_2021[time_col].dt.date

# 5. Aggregate Hourly to Daily
daily_weather = weather_2021.groupby('date_only').agg({
    temp_col: 'mean',
    precip_col: 'max'
}).reset_index()

# 6. Create ML Features (Fixed variable error)
daily_weather['is_rainy'] = (daily_weather[precip_col] > 0).astype(int)
daily_weather.rename(columns={temp_col: 'temperature'}, inplace=True)

# 7. Merge with taxi data
taxi_df['pickup_date'] = pd.to_datetime(taxi_df['tpep_pickup_datetime']).dt.date

taxi_df = taxi_df.merge(
    daily_weather[['date_only', 'is_rainy', 'temperature']],
    how='left',
    left_on='pickup_date',
    right_on='date_only'
)

# 8. Clean up and Fill
if 'date_only' in taxi_df.columns:
    taxi_df.drop(columns=['date_only'], inplace=True)

taxi_df['is_rainy'] = taxi_df['is_rainy'].fillna(0).astype(int)
taxi_df['temperature'] = taxi_df['temperature'].fillna(taxi_df['temperature'].median())

print(f"✅ Weather integration successful! Final rows: {len(taxi_df):,}")

In [ ]:
print("⚡ Optimizing memory types...")
initial_mem = taxi_df.memory_usage().sum() / 1e6

# 1. Convert Boroughs and Zones to Category (Saves HUGE amounts of RAM)
cat_cols = ['pickup_borough', 'pickup_zone', 'dropoff_borough', 'dropoff_zone']
for col in cat_cols:
    if col in taxi_df.columns:
        taxi_df[col] = taxi_df[col].astype('category')

# 2. Downcast Floats
float_cols = ['trip_distance', 'fare_amount', 'trip_duration_minutes', 'avg_speed_mph', 'temperature']
for col in float_cols:
    if col in taxi_df.columns:
        taxi_df[col] = pd.to_numeric(taxi_df[col], downcast='float')

# 3. Downcast Integers
int_cols = ['PULocationID', 'DOLocationID', 'passenger_count', 'is_rainy', 'pickup_hour']
for col in int_cols:
    if col in taxi_df.columns:
        taxi_df[col] = pd.to_numeric(taxi_df[col], downcast='integer')

final_mem = taxi_df.memory_usage().sum() / 1e6
gc.collect()

print(f"✅ Memory reduced from {initial_mem:.1f} MB to {final_mem:.1f} MB ({(1-final_mem/initial_mem)*100:.1f}% reduction)")

In [ ]:
processed_file = os.path.join(PROCESSED_DATA_DIR, "urbanflow_clean_2021.parquet")
taxi_df.to_parquet(processed_file, index=False)
print(f"✅ Success! Processed dataset saved to: {processed_file}")

In [ ]:
#EDA

print("📊 Performing Statistical Analysis...")

# 1. Comparison: Rain vs No Rain
rain_stats = taxi_df.groupby('is_rainy')['trip_duration_minutes'].agg(['mean', 'median', 'std']).reset_index()
print("\nImpact of Rain on Trip Duration:")
print(rain_stats)

# 2. Temporal Distribution
plt.figure(figsize=(12, 5))
sns.lineplot(data=taxi_df.sample(100000), x='pickup_hour', y='trip_duration_minutes', hue='is_rainy')
plt.title("Average Trip Duration by Hour (Rainy vs Clear)", fontsize=14, fontweight='bold')
plt.xlabel("Hour of Day")
plt.ylabel("Minutes")
plt.show()

# 3. Borough Distribution
plt.figure(figsize=(10, 5))
taxi_df['pickup_borough'].value_counts().plot(kind='bar', color='skyblue')
plt.title("Trips per Borough", fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#Refined Feature Selection

# Select features based on research goals
features = [
    'trip_distance', 'pickup_hour', 'pickup_dayofweek',
    'is_weekend', 'is_rush_hour', 'temperature', 'is_rainy',
    'pickup_borough'
]

# Filtering for the most significant boroughs to avoid "Sparse Matrix" memory blow-up
main_boroughs = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx']
ml_df = taxi_df[taxi_df['pickup_borough'].isin(main_boroughs)].copy()

X = ml_df[features]
y = ml_df['trip_duration_minutes']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Prepared {len(X_train):,} training samples and {len(X_test):,} test samples.")

In [ ]:
#Preprocessing Pipeline

from sklearn.preprocessing import RobustScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['pickup_borough']),
        ('num', RobustScaler(), ['trip_distance', 'temperature', 'pickup_hour'])
    ],
    remainder='passthrough'
)
print("✅ Preprocessing pipeline ready.")

In [ ]:
#Model

import lightgbm as lgb

print("🚀 Training LightGBM Regressor (High Efficiency)...")

model_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"🏁 Model Results -> RMSE: {rmse:.2f} mins | R²: {r2:.3f}")

In [ ]:
from sklearn.metrics import mean_absolute_error

# 1. Calculate final metrics
mae = mean_absolute_error(y_test, y_pred)
print("\n" + "="*40)
print("🏆 FINAL MODEL PERFORMANCE")
print("="*40)
print(f"Mean Absolute Error: {mae:.2f} minutes")
print(f"RMSE: {rmse:.2f} minutes")
print(f"R² Score: {r2:.4f}")
print("="*40)

# 2. Visualize Predictions vs Reality (Sample for speed)
plt.figure(figsize=(10, 6))
plt.scatter(y_test[:1000], y_pred[:1000], alpha=0.3, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title("Actual vs. Predicted Trip Duration", fontsize=14)
plt.xlabel("Actual Duration (min)")
plt.ylabel("Predicted Duration (min)")
plt.show()

# 3. Save the best model (CRITICAL)
model_save_path = os.path.join(MODELS_DIR, 'urbanflow_lgbm_model.pkl')
joblib.dump(model_pipeline, model_save_path)
print(f"\n✅ Model successfully saved to Drive: {model_save_path}")

In [ ]:
print("🗺️ Generating Spatial Flow Analysis...")

# 1. Filter for the most active borough (Manhattan) for a clear visual
map_data = taxi_df[taxi_df['pickup_borough'] == 'Manhattan'].sample(100000)

# 2. Create a hexbin plot to show trip density
# This shows the flow from Pickup (X) to Dropoff (Y)
plt.figure(figsize=(12, 10))
hb = plt.hexbin(map_data['PULocationID'], map_data['DOLocationID'],
               gridsize=40, cmap='inferno', mincnt=5)
cb = plt.colorbar(hb)
cb.set_label('Concentration of Trips')

plt.title("Urban Mobility Flow: Pickup vs. Dropoff Hotspots", fontsize=16, fontweight='bold')
plt.xlabel("Pickup Location ID (Zone Index)", fontsize=12)
plt.ylabel("Dropoff Location ID (Zone Index)", fontsize=12)

# Annotate hotspots
plt.annotate('Highest Flow Density', xy=(map_data['PULocationID'].median(), map_data['DOLocationID'].median()),
             xytext=(200, 250), arrowprops=dict(facecolor='white', shrink=0.05))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'spatial_flow_hexbin.png'), dpi=300)
plt.show()

print(f"✅ Spatial flow map saved to: {OUTPUTS_DIR}")

In [ ]:
#Feature Importance

# Extract feature importance from LightGBM
lgbm_model = model_pipeline.named_steps['regressor']
# Get feature names after preprocessing
ohe_cols = model_pipeline.named_steps['preprocess'].named_transformers_['cat'].get_feature_names_out(['pickup_borough']).tolist()
feature_names = ohe_cols + ['trip_distance', 'temperature', 'pickup_hour', 'pickup_dayofweek', 'is_weekend', 'is_rush_hour', 'is_rainy']

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': lgbm_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', palette='magma')
plt.title("Top 10 Drivers of Trip Duration (Feature Importance)", fontsize=14)
plt.show()

In [ ]:
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import gc

print("⚙️ Preparing data for XGBoost...")

# 1. Define Features (Avoiding Data Leakage)
# We exclude 'fare_amount', 'avg_speed_mph', and timestamps
features = [
    'trip_distance',
    'PULocationID',
    'DOLocationID',
    'pickup_hour',
    'pickup_dayofweek',
    'pickup_month',
    'is_weekend',
    'is_rush_hour',
    'temperature',
    'is_rainy',
    'pickup_borough',
    'dropoff_borough'
]

target = 'trip_duration_minutes'

# 2. Create Feature Matrix (X) and Target (y)
# We use .copy() to ensure we have a clean dataframe slice
X = taxi_df[features].copy()
y = taxi_df[target].copy()

# 3. Handle Categorical Types for Sklearn Compatibility
# Ensure borough columns are strings/categories for the encoder
X['pickup_borough'] = X['pickup_borough'].astype(str)
X['dropoff_borough'] = X['dropoff_borough'].astype(str)

print(f"✅ Features selected: {len(features)}")
print(f"✅ Input shape: {X.shape}")

# 4. Train/Test Split (80% Train, 20% Test)
print("✂️ Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Free up memory
gc.collect()
print(f"✅ Training samples: {len(X_train):,}")
print(f"✅ Test samples: {len(X_test):,}")

In [ ]:
print("🔧 Building XGBoost Pipeline...")

# 1. Define Preprocessing Steps
# - OneHotEncoder for Boroughs (Low cardinality, safe for RAM)
# - Passthrough for LocationIDs and other numericals (XGBoost handles these well)
categorical_cols = ['pickup_borough', 'dropoff_borough']
numeric_cols = [c for c in features if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ],
    remainder='passthrough' # Keep any other columns if missed
)

# 2. Define the XGBoost Model
# tree_method='hist' is CRITICAL for speed on large datasets (9M rows)
xgb_model = xgb.XGBRegressor(
    n_estimators=200,        # Number of trees
    learning_rate=0.1,       # Step size optimization
    max_depth=10,            # Depth of trees (controls complexity)
    subsample=0.8,           # Use 80% of data per tree (prevents overfitting)
    colsample_bytree=0.8,    # Use 80% of features per tree
    tree_method='hist',      # Fast histogram optimized algorithm
    n_jobs=-1,               # Use all CPU cores
    random_state=42
)

# 3. Bundle into Pipeline
model_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', xgb_model)
])

print("✅ Pipeline constructed successfully.")

In [ ]:
import time

print("🚀 Starting XGBoost Training...")
start_time = time.time()

# Fit the model
model_pipeline.fit(X_train, y_train)

end_time = time.time()
print(f"✅ Training complete in {(end_time - start_time)/60:.2f} minutes.")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import numpy as np
import os # Ensure os is imported for path joining

print("📊 Evaluating XGBoost Performance...")

# 1. Make Predictions using the trained pipeline
y_pred_xgb = model_pipeline.predict(X_test)

# 2. Calculate Metrics
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("\n" + "="*40)
print("🥈 XGBoost FINAL RESULTS")
print("="*40)
print(f"RMSE (Root Mean Sq Error): {rmse_xgb:.2f} minutes")
print(f"MAE (Mean Absolute Error): {mae_xgb:.2f} minutes")
print(f"R² Score (Accuracy):       {r2_xgb:.4f}")
print("="*40)

# 3. Save the Model (Crucial for your App later)
model_path = os.path.join(MODELS_DIR, 'urbanflow_xgboost_final.pkl')
joblib.dump(model_pipeline, model_path)
print(f"\n✅ Model saved successfully to: {model_path}")

In [ ]:
# Extract feature names and importance
model = model_pipeline.named_steps['model']
preprocessor = model_pipeline.named_steps['preprocess']

# Get names from OneHotEncoder
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
# Combine with numeric names
feature_names = list(cat_names) + numeric_cols

# Create DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Factors Affecting NYC Trip Duration (XGBoost)', fontsize=15, fontweight='bold')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'xgboost_feature_importance.png'))
plt.show()

In [ ]:
import gc
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor

print("🛡️ Initializing Crash-Proof Training Sequence...")

# ==========================================
# PHASE 1: AGGRESSIVE MEMORY CLEANUP
# ==========================================
# If taxi_df exists in memory from previous cells, KILL IT.
if 'taxi_df' in globals():
    del taxi_df
    print("🗑️ Deleted 'taxi_df' to free RAM.")

if 'X' in globals():
    del X
if 'y' in globals():
    del y

gc.collect() # Force Python to release memory immediately
print("✅ RAM Cleaned.")

# ==========================================
# PHASE 2: INTELLIGENT DATA LOADING
# ==========================================
# Check if X_train exists. If not (due to crash), reload it.
if 'X_train' not in globals():
    print("🔄 Data missing (Crash detected). Reloading from Drive...")

    # Paths (Redefine just in case)
    BASE_DIR = "/content/drive/MyDrive/UrbanFlow"
    PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data/processed")
    processed_file = os.path.join(PROCESSED_DATA_DIR, "urbanflow_clean_2021.parquet")

    # Load
    temp_df = pd.read_parquet(processed_file)

    # Select Features
    features = [
        'trip_distance', 'pickup_hour', 'pickup_dayofweek', 'pickup_month',
        'is_weekend', 'is_rush_hour', 'temperature', 'is_rainy',
        'pickup_borough', 'dropoff_borough',
        'PULocationID', 'DOLocationID'
    ]
    target = 'trip_duration_minutes'

    X = temp_df[features].copy()
    y = temp_df[target].copy()

    # Delete temp_df immediately
    del temp_df
    gc.collect()

    # Split
    print("✂️ Re-splitting data...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Delete X and y immediately
    del X, y
    gc.collect()
    print("✅ Data Loaded & Split.")

else:
    print("✅ Using existing X_train from memory.")

# ==========================================
# PHASE 3: CATBOOST TRAINING
# ==========================================
print("🐱 Training CatBoost Regressor (The Specialist)...")

# Identify Categorical Features (Robust check for object OR category)
# CatBoost needs to know which columns are boroughs
cat_features_indices = [
    i for i, col in enumerate(X_train.columns)
    if X_train[col].dtype.name in ['object', 'category']
]
print(f"Categorical Indices: {cat_features_indices}")

# Define Model
cb_model = CatBoostRegressor(
    iterations=200,
    learning_rate=0.1,
    depth=8,                # Depth 8 is the Safe Sweet Spot for Colab
    loss_function='RMSE',
    random_seed=42,
    verbose=50,             # Print every 50 steps
    allow_writing_files=False,
    task_type="CPU"
)

# Train
cb_model.fit(
    X_train, y_train,
    cat_features=cat_features_indices,
    eval_set=(X_test, y_test),
    early_stopping_rounds=20
)

# ==========================================
# PHASE 4: EVALUATION & SAVE
# ==========================================
print("📊 Evaluating...")
y_pred_cb = cb_model.predict(X_test)

rmse_cb = np.sqrt(mean_squared_error(y_test, y_pred_cb))
r2_cb = r2_score(y_test, y_pred_cb)

print("\n" + "="*40)
print("🥉 CatBoost PERFORMANCE")
print("="*40)
print(f"RMSE: {rmse_cb:.2f} minutes")
print(f"R²:   {r2_cb:.4f}")
print("="*40)

# Save
model_save_path = os.path.join(BASE_DIR, "models/urbanflow_catboost.cbm")
cb_model.save_model(model_save_path)
print(f"✅ CatBoost model saved to: {model_save_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

print("📊 Generating Model Comparison Chart...")

# ==========================================
# 1. GATHER METRICS
# ==========================================
# If your variables exist in memory, we use them.
# If not (e.g., after a crash), we use the values you shared in the chat.
try:
    # Try to grab variables from previous cells
    scores_rmse = [rmse, rmse_xgb, rmse_cb]
    scores_r2 = [r2, r2_xgb, r2_cb]
    print("✅ Using calculated metrics from memory.")
except NameError:
    # ⚠️ MANUAL OVERRIDE: If variables are missing, type your results here
    print("⚠️ Variables missing. Using saved log values (You can edit these lines).")

    # Replace these numbers with the exact output you saw!
    scores_rmse = [4.76, 4.70, 4.62]  # LightGBM, XGBoost, CatBoost
    scores_r2   = [0.805, 0.810, 0.815]

models = ['LightGBM', 'XGBoost', 'CatBoost']

# ==========================================
# 2. CREATE THE PLOT
# ==========================================
x = np.arange(len(models))
width = 0.35

fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot RMSE (Bar Chart)
rects1 = ax1.bar(x - width/2, scores_rmse, width,
                 label='RMSE (Lower is Better)',
                 color=['#3498db', '#2ecc71', '#9b59b6'])

ax1.set_ylabel('RMSE (Minutes)', fontweight='bold', fontsize=12)
ax1.set_title('The Battle of Boosting: Model Comparison', fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=12)
ax1.set_ylim(4.0, 5.0) # Zoom in to see differences
ax1.legend(loc='upper left')

# Plot R2 (Line Chart)
ax2 = ax1.twinx()
ax2.plot(x, scores_r2, color='#e74c3c', marker='D', markersize=10, linewidth=3, label='R² Score')
ax2.set_ylabel('R² Score (Higher is Better)', fontweight='bold', fontsize=12, color='#e74c3c')
ax2.legend(loc='upper right')

# Add Labels on Bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax1.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold')

autolabel(rects1)

# ==========================================
# 3. SAVE TO FILE
# ==========================================
# Create outputs folder if it doesn't exist
output_dir = "/content/drive/MyDrive/UrbanFlow/outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

save_path = os.path.join(output_dir, 'big_three_showdown.png')
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.show()

print(f"✅ Comparison chart saved to: {save_path}")
print("⬇️ PLEASE DOWNLOAD THIS FILE NOW for your App.")